# NFL Elo Rating Model

Prospective NFL Elo ratings with margin-of-victory adjustment, home-field advantage, offseason regression, and expanding-window parameter tuning.

In [86]:
# pip install nflreadpy pandas pyarrow


## 1. Setup and NFL Data

In [25]:
import numpy as np
import pandas as pd
import nflreadpy as nfl
from sklearn.metrics import accuracy_score, log_loss

#Load games from nfl library
games = nfl.load_schedules(seasons=True).to_pandas()
print(games.columns)
print(games.shape)
print(games.head())



Index(['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday',
       'gametime', 'away_team', 'away_score', 'home_team', 'home_score',
       'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis',
       'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest',
       'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds',
       'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game',
       'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id',
       'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee',
       'stadium_id', 'stadium'],
      dtype='object')
(7548, 46)
           game_id  season game_type  week     gameday weekday gametime  \
0  1999_01_MIN_ATL    1999       REG     1  1999-09-12  Sunday     None   
1   1999_01_KC_CHI    1999       REG     1  1999-09-12  Sunday     None   
2  1999_01_PIT_CLE    1999       REG     1  1999-09-12  Sunday     None   
3   1999_01_OAK_GB    1999      

In [ ]:
## 2. Core Elo Rating Functions

In [47]:
import numpy as np
import pandas as pd
from IPython.display import display


def null_coalesce(x, y):
    return y if x is None else x

def compress_exp_grid(min_val, max_val, n=10, curve=2):
    grid = np.unique(np.round(
        min_val + (max_val - min_val) *
        (np.exp(curve * np.linspace(0, 1, n)) - 1) / (np.exp(curve) - 1)
    ))
    return grid

def calc_mov_scale(games):
    mean_log_margin = np.log(
        (games["home_score"] - games["away_score"]).abs() + 1
    ).mean()
    return mean_log_margin

def nfl_home_result(home_score, away_score, overtime):
    conditions = [
        (home_score > away_score) & (overtime == 0),
        (home_score < away_score) & (overtime == 0),
        (home_score > away_score) & (overtime == 1),
        (home_score < away_score) & (overtime == 1),
        home_score == away_score
    ]
    choices = [1.00, 0.00, 0.75, 0.25, 0.50]
    return np.select(conditions, choices, default=np.nan)

def elo_season(games, K, ratings=None, scale=400, start_rating=1000,
               use_mov=True, mov_scale=1, use_home_adv=True, home_adv=0):

    ratings = {} if ratings is None else ratings.copy()
    n = len(games)

    if use_mov and (pd.isna(mov_scale) or mov_scale <= 0):
        mov_scale = 1
    if not use_home_adv:
        home_adv = 0

    elo_home_pre = np.zeros(n)
    elo_away_pre = np.zeros(n)
    elo_home_post = np.zeros(n)
    elo_away_post = np.zeros(n)
    p_home = np.zeros(n)

    for i in range(n):
        g = games.iloc[i]

        hid = str(g["home_team"])
        aid = str(g["away_team"])

        Ra = ratings.get(hid, start_rating)
        Rb = ratings.get(aid, start_rating)

        elo_home_pre[i] = Ra
        elo_away_pre[i] = Rb

        p = 1 / (1 + 10 ** ((Rb - Ra - home_adv) / scale))
        p_home[i] = p

        y = nfl_home_result(
            home_score=g["home_score"],
            away_score=g["away_score"],
            overtime=g["overtime"]
        )

        if pd.isna(y):
            raise ValueError(f"Invalid NFL result coding for game_id: {g['game_id']}")

        point_diff = abs(g["home_score"] - g["away_score"])
        if g["home_score"] > g["away_score"]:
          winner_elo_diff = (Ra + home_adv) - Rb
        elif g["home_score"] < g["away_score"]:
          winner_elo_diff = Rb - (Ra + home_adv)
        else:
          winner_elo_diff = 0

        if not use_mov:
          mov_mult = 1
        elif g["home_score"] == g["away_score"]:
          mov_mult = 1
        else:
          mov_mult = np.log(point_diff + 1) * (2.2 / (0.001 * winner_elo_diff + 2.2)) / mov_scale

        delta = K * (y - p) * mov_mult

        ratings[hid] = Ra + delta
        ratings[aid] = Rb - delta

        elo_home_post[i] = ratings[hid]
        elo_away_post[i] = ratings[aid]

    games_out = games.copy()
    games_out["elo_home_pre"] = elo_home_pre
    games_out["elo_away_pre"] = elo_away_pre
    games_out["elo_home_post"] = elo_home_post
    games_out["elo_away_post"] = elo_away_post
    games_out["p_home"] = p_home
    games_out["p_away"] = 1 - p_home

    return {"games": games_out, "ratings": ratings}

### Single-Season Elo Sanity Check

In [48]:
# Sort it chronologically, calculate its MOV scale, and run
test_games = games[
    (games["season"] == 2024) &
    games["home_score"].notna() &
    games["away_score"].notna()
].copy()

test_games = test_games.sort_values(["gameday", "game_id"]).reset_index(drop=True)

mov_scale = calc_mov_scale(test_games)

test_elo = elo_season(
    games=test_games,
    K=20,
    ratings={},
    scale=400,
    start_rating=1000,
    use_mov=True,
    mov_scale=mov_scale,
    use_home_adv=True,
    home_adv=50
)

test_elo["games"][
    ["gameday", "home_team", "away_team",
     "elo_home_pre", "elo_away_pre",
     "p_home", "elo_home_post", "elo_away_post"]
].head(20)

display(test_elo["ratings"])

#Sanity Check - average of elo values should still be around starting mean of 1000
print("Average ELO Rank: ",
      sum(test_elo["ratings"].values())
      / len(test_elo["ratings"]))

{'KC': np.float64(1082.1410739289765),
 'BAL': np.float64(1085.0029964991543),
 'PHI': np.float64(1136.6408942963722),
 'GB': np.float64(1048.6897287643872),
 'BUF': np.float64(1091.4035283122469),
 'ARI': np.float64(990.4287605298258),
 'NO': np.float64(940.4030492884059),
 'CAR': np.float64(916.1109742111105),
 'CLE': np.float64(900.5990508042443),
 'DAL': np.float64(960.9313654303023),
 'SEA': np.float64(1017.0539019733426),
 'DEN': np.float64(1040.6456537309425),
 'IND': np.float64(967.2698647453412),
 'HOU': np.float64(1019.3156627426816),
 'MIA': np.float64(989.0127740341378),
 'JAX': np.float64(936.9866554763992),
 'DET': np.float64(1098.161997374354),
 'LA': np.float64(1023.4345096424472),
 'LAC': np.float64(1040.0924434757628),
 'LV': np.float64(918.7375564290048),
 'NYG': np.float64(909.1271341245094),
 'MIN': np.float64(1060.2498979434017),
 'CIN': np.float64(1024.128651789547),
 'NE': np.float64(934.2967345444166),
 'ATL': np.float64(979.5072187351919),
 'PIT': np.float64(1

Average ELO Rank:  1000.0


## 3. Offseason Regression and Evaluation

In [49]:
def carry_over(ratings, carry=0.6):
    if len(ratings) == 0:
        return ratings

    mu = np.mean(list(ratings.values()))
    return {team: carry * rating + (1 - carry) * mu
            for team, rating in ratings.items()}

def logloss(p, y, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(y * np.log(p) + (1 - y) * np.log1p(-p))

### Carryover Sanity Check

In [50]:
#Sanity Checks
test_carry = carry_over(test_elo["ratings"], carry=0.75)

print(np.mean(list(test_carry.values())))
print(max(test_elo["ratings"].values()))
print(max(test_carry.values()))

1000.0
1136.6408942963722
1102.4806707222792


## 4. Elo Parameter Evaluation

In [52]:
def score_elo_params(all_games, seasons_tune, K, home_adv,
                     use_mov=True, mov_scale=1, use_home_adv=True,
                     scale=400, start_rating=1000, carry=0.6):

  ratings = {}
  ll_vec = []

  if not use_home_adv:
      home_adv = 0

  for s in seasons_tune:
      g_s = (all_games[all_games["season"] == s]
              .sort_values(["gameday", "game_id"])
              .reset_index(drop=True))

      res = elo_season(games=g_s, ratings=ratings, K=K, scale=scale,
                       start_rating=start_rating, use_mov=use_mov,
                       mov_scale=mov_scale, use_home_adv=use_home_adv,
                       home_adv=home_adv)

      games_res = res["games"]
      non_ties = games_res["home_score"] != games_res["away_score"]

      p = games_res.loc[non_ties, "p_home"].to_numpy()
      y = (games_res.loc[non_ties, "home_score"] >
            games_res.loc[non_ties, "away_score"]).astype(float).to_numpy()

      ll_vec.extend(logloss(p, y))
      ratings = carry_over(res["ratings"], carry=carry)

  return np.nanmean(ll_vec)

### Initial Parameter Test

In [53]:
test_score = score_elo_params(
    all_games=games,
    seasons_tune=[2021, 2022, 2023, 2024],
    K=20,
    home_adv=50,
    use_mov=True,
    mov_scale=calc_mov_scale(games[games["season"].isin([2021, 2022, 2023, 2024])]),
    use_home_adv=True,
    scale=400,
    start_rating=1000,
    carry=0.75
)

print(test_score)

0.6503069250902233


$0.6503$ is already meaningfully better than a naive 50/50 baseline:
$−log(.5)=0.6931$

## 5. Hyperparameter Tuning

In [55]:
from itertools import product

def tune_elo_params(all_games, seasons_tune, K_grid, home_adv_grid,
                    use_mov=True, mov_scale=1, use_home_adv=True,
                    scale=400, start_rating=1000, carry=0.6,
                    use_carry_grid=True,
                    carryover_grid=(0.35, 0.50, 0.65, 0.80)):

    if not use_home_adv:
        home_adv_grid = [0]

    carry_grid = carryover_grid if use_carry_grid else [carry]
    results = []

    for K, home_adv, carry_val in product(K_grid, home_adv_grid, carry_grid):
        ll = score_elo_params(
            all_games=all_games, seasons_tune=seasons_tune,
            K=K, home_adv=home_adv,
            use_mov=use_mov, mov_scale=mov_scale,
            use_home_adv=use_home_adv, scale=scale,
            start_rating=start_rating, carry=carry_val
        )

        results.append({"K": K, "home_adv": home_adv, "carry": carry_val,
                        "logloss": ll})

    results = pd.DataFrame(results).sort_values("logloss").reset_index(drop=True)

    best = results.iloc[0]

    return {"K_best": best["K"], "home_adv_best": best["home_adv"],
            "carry_best": best["carry"], "logloss": best["logloss"],
            "results": results}

### Tuning Sanity Check

In [56]:
K_grid = [10, 15, 20, 25]
home_adv_grid = [25, 40, 55, 70]

mov_scale = calc_mov_scale(
    games[games["season"].isin([2021, 2022, 2023, 2024])]
)

test_tune = tune_elo_params(
    all_games=games,
    seasons_tune=[2021, 2022, 2023, 2024],
    K_grid=K_grid,
    home_adv_grid=home_adv_grid,
    mov_scale=mov_scale,
    carryover_grid=[0.50, 0.65, 0.75, 0.85]
)

print(test_tune["K_best"])
print(test_tune["home_adv_best"])
print(test_tune["carry_best"])
print(test_tune["logloss"])

test_tune["results"].head(10)

25.0
40.0
0.85
0.6463982500974149


,K,home_adv,carry,logloss
0,25,40,0.85,0.646398
1,25,25,0.85,0.646476
2,25,40,0.75,0.646507
3,25,25,0.75,0.646573
4,25,40,0.65,0.647088
5,25,25,0.65,0.647146
6,25,55,0.85,0.648168
7,25,55,0.75,0.648294
8,25,40,0.50,0.648652
9,25,25,0.50,0.648702


## 6. Prospective Expanding-Window Elo

In [ ]:
def run_expanding_k_elo(all_games, first_train_year=2012, first_eval_year=2019,
                        K_grid=None, home_adv_grid=None, scale=400,
                        start_rating=1000, carry=0.6, use_carry_grid=True,
                        carryover_grid=(0.35, 0.50, 0.65, 0.80),
                        param_tune_window=None, use_mov=True, use_home_adv=True):

    if K_grid is None:
        K_grid = compress_exp_grid(1, 50, n=10, curve=3)
    if home_adv_grid is None:
        home_adv_grid = compress_exp_grid(-4, 40, n=10, curve=2)
    if not use_home_adv:
        home_adv_grid = [0]

    K_grid = np.atleast_1d(K_grid)
    home_adv_grid = np.atleast_1d(home_adv_grid)
    fixed_param_mode = len(K_grid) == 1 and len(home_adv_grid) == 1 and not use_carry_grid

    seasons = sorted(all_games["season"].unique())
    eval_years = [s for s in seasons if s >= first_eval_year]

    all_out = []
    param_out = []

    for yr in eval_years:
        train_years = [s for s in seasons if first_train_year <= s < yr]
        if len(train_years) == 0:
            raise ValueError(f"No training seasons available before eval year: {yr}")

        param_tune_years = (
            train_years if param_tune_window is None
            else train_years[-param_tune_window:]
        )

        train_games = all_games[all_games["season"].isin(train_years)]
        mov_scale = calc_mov_scale(train_games) if use_mov else np.nan

        if fixed_param_mode:
            K_best = K_grid[0]
            home_adv_best = home_adv_grid[0] if use_home_adv else 0
            carry_best = carry
            best_logloss = np.nan
        else:
            param_fit = tune_elo_params(
                all_games=all_games, seasons_tune=param_tune_years,
                K_grid=K_grid, home_adv_grid=home_adv_grid,
                use_mov=use_mov, mov_scale=mov_scale,
                use_home_adv=use_home_adv, scale=scale,
                start_rating=start_rating, carry=carry,
                use_carry_grid=use_carry_grid,
                carryover_grid=carryover_grid
            )

            K_best = param_fit["K_best"]
            home_adv_best = param_fit["home_adv_best"] if use_home_adv else 0
            carry_best = param_fit["carry_best"] if use_carry_grid else carry
            best_logloss = param_fit["logloss"]

        ratings = {}

        for s in train_years:
            season_games = (all_games[all_games["season"] == s]
                            .sort_values(["gameday", "gametime", "game_id"])
                            .reset_index(drop=True))

            res = elo_season(
                games=season_games, ratings=ratings, K=K_best,
                scale=scale, start_rating=start_rating,
                use_mov=use_mov, mov_scale=mov_scale,
                use_home_adv=use_home_adv, home_adv=home_adv_best
            )

            ratings = carry_over(res["ratings"], carry=carry_best)

        eval_games = (all_games[all_games["season"] == yr]
                      .sort_values(["gameday", "gametime", "game_id"])
                      .reset_index(drop=True))

        res_eval = elo_season(
            games=eval_games, ratings=ratings, K=K_best,
            scale=scale, start_rating=start_rating,
            use_mov=use_mov, mov_scale=mov_scale,
            use_home_adv=use_home_adv, home_adv=home_adv_best
        )

        games_out = res_eval["games"].copy()
        games_out["K_used"] = K_best
        games_out["home_adv_used"] = home_adv_best
        games_out["carry_used"] = carry_best
        games_out["mov_scale_used"] = mov_scale
        games_out["use_mov"] = use_mov
        games_out["use_home_adv"] = use_home_adv
        games_out["train_start"] = min(train_years)
        games_out["train_end"] = max(train_years)
        games_out["param_tune_start"] = min(param_tune_years)
        games_out["param_tune_end"] = max(param_tune_years)
        games_out["param_tune_n"] = len(param_tune_years)
        games_out["param_tune_window"] = param_tune_window

        all_out.append(games_out)

        param_out.append({
            "eval_year": yr,
            "train_start": min(train_years),
            "train_end": max(train_years),
            "param_tune_start": min(param_tune_years),
            "param_tune_end": max(param_tune_years),
            "param_tune_n": len(param_tune_years),
            "param_tune_window": param_tune_window,
            "K_best": K_best,
            "home_adv_best": home_adv_best,
            "carry_best": carry_best,
            "mov_scale": mov_scale,
            "use_mov": use_mov,
            "use_home_adv": use_home_adv,
            "use_carry_grid": use_carry_grid,
            "logloss": best_logloss
        })

    return {
        "games": pd.concat(all_out, ignore_index=True),
        "K_by_year": pd.DataFrame(param_out)
    }

### Prospective Validation Test

In [81]:
# Exclude Preseason and Unplayed Games
elo_df = games[games["game_type"].isin(["REG", "WC", "DIV", "CON", "SB"]) &
               games["home_score"].notna() &
               games["away_score"].notna()].copy()

test_expanding = run_expanding_k_elo(
    all_games=elo_df,
    first_train_year=2018,
    first_eval_year=2023,
    K_grid=[15, 20, 25],
    home_adv_grid=[25, 40, 55],
    scale=400,
    start_rating=1000,
    use_carry_grid=True,
    carryover_grid=[0.3, 0.5, 0.75, 0.90],
    param_tune_window=None,
    use_mov=True,
    use_home_adv=True
)

#Inspect
display(test_expanding["K_by_year"])
display(test_expanding["games"][["season", "gameday", "home_team", "away_team",
                         "elo_home_pre", "elo_away_pre", "p_home", "K_used",
                         "home_adv_used", "carry_used", "mov_scale_used"]].head())

#Sanity Check
test_expanding["K_by_year"][["eval_year", "train_start", "train_end",
                             "param_tune_start", "param_tune_end"]]

,eval_year,train_start,train_end,param_tune_start,param_tune_end,param_tune_n,param_tune_window,K_best,home_adv_best,carry_best,mov_scale,use_mov,use_home_adv,use_carry_grid,logloss
0,2023,2018,2022,2018,2022,5,None,25.0,25.0,0.75,2.211741,True,True,True,0.641638
1,2024,2018,2023,2018,2023,6,None,25.0,25.0,0.75,2.211388,True,True,True,0.644038
2,2025,2018,2024,2018,2024,7,None,25.0,25.0,0.75,2.212730,True,True,True,0.639921
3,2026,2018,2025,2018,2025,8,None,25.0,25.0,0.75,2.209776,True,True,True,0.640277


,season,gameday,home_team,away_team,elo_home_pre,elo_away_pre,p_home,K_used,home_adv_used,carry_used,mov_scale_used
0,2023,2023-09-07,KC,DET,1143.437059,974.549985,0.753265,25.0,25.0,0.75,2.211741
1,2023,2023-09-10,WAS,ARI,972.622526,945.187375,0.574893,25.0,25.0,0.75,2.211741
2,2023,2023-09-10,ATL,CAR,955.066586,965.250314,0.521309,25.0,25.0,0.75,2.211741
3,2023,2023-09-10,CLE,CIN,988.981861,1103.527018,0.373914,25.0,25.0,0.75,2.211741
4,2023,2023-09-10,BAL,HOU,1026.001226,886.395049,0.720622,25.0,25.0,0.75,2.211741


,eval_year,train_start,train_end,param_tune_start,param_tune_end
0,2023,2018,2022,2018,2022
1,2024,2018,2023,2018,2023
2,2025,2018,2024,2018,2024
3,2026,2018,2025,2018,2025


### Out-of-Sample Performance

In [128]:
#Calculate out-of-sample performance of 2023-2026
eval_games = test_expanding["games"].copy()
non_ties = eval_games["home_score"] != eval_games["away_score"]

eval_games["actual_home_win"] = (
    eval_games["home_score"] > eval_games["away_score"]).astype(int)

eval_games["pred_home_win"] = (eval_games["p_home"] >= 0.5).astype(int)

eval_games["logloss"] = np.nan
eval_games.loc[non_ties, "logloss"] = logloss(
    eval_games.loc[non_ties, "p_home"],
    eval_games.loc[non_ties, "actual_home_win"]
)

performance_by_year = (eval_games.loc[non_ties].groupby("season").agg(
    games=("game_id", "size"),
    accuracy=("pred_home_win",
              lambda x: (x == eval_games.loc[x.index, "actual_home_win"]).mean()),
    logloss=("logloss", "mean")).reset_index())

print(performance_by_year)

   season  games  accuracy   logloss
0    2023    285  0.617544  0.655543
1    2024    285  0.677193  0.616031
2    2025    284  0.640845  0.642841
3    2026      2  0.500000  0.686665


In [84]:
overall_accuracy = (eval_games.loc[non_ties, "pred_home_win"] ==
                    eval_games.loc[non_ties, "actual_home_win"]).mean()

overall_logloss = eval_games.loc[non_ties, "logloss"].mean()

print("Accuracy:", overall_accuracy)
print("Log Loss:", overall_logloss)

Accuracy: 0.6448598130841121
Log Loss: 0.6382464576517052


In [80]:
accuracy_by_year = (
    games_eval.loc[non_ties]
    .groupby("season")
    .apply(lambda x: (x["pred_home_win"] == x["actual_home_win"]).mean())
    .reset_index(name="accuracy")
)

accuracy_by_year

/tmp/ipykernel_2232/2648429389.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: (x["pred_home_win"] == x["actual_home_win"]).mean())


,season,accuracy
0,2023,0.617544
1,2024,0.677193
2,2025,0.640845
3,2026,0.500000


## 7. Final Prospective Elo Model

In [85]:
nfl_elo = run_expanding_k_elo(all_games=elo_df, first_train_year=2018,
                    first_eval_year=2023,
                    K_grid = compress_exp_grid(10, 50, n=10, curve=2),
                    home_adv_grid = compress_exp_grid(0, 60, n=10, curve=2),
                    scale=400, start_rating=1000, use_carry_grid=True,
                    carryover_grid = [0.5, 0.65, 0.75, 0.85, 0.90],
                    param_tune_window=None, use_mov=True, use_home_adv=True)


### Performance Check

In [88]:
eval_games = nfl_elo["games"].query("home_score != away_score").copy()

eval_games["actual"] = (eval_games["home_score"] > eval_games["away_score"]).astype(int)
eval_games["pred"] = (eval_games["p_home"] >= 0.5).astype(int)
eval_games["correct"] = eval_games["pred"] == eval_games["actual"]
eval_games["ll"] = -(eval_games["actual"] * np.log(eval_games["p_home"]) + (1 - eval_games["actual"]) * np.log1p(-eval_games["p_home"]))

print("Accuracy:", accuracy_score(eval_games["actual"], eval_games["pred"]))
print("Log Loss:", log_loss(eval_games["actual"], eval_games["p_home"]))

Accuracy: 0.6483644859813084
Log Loss: 0.6341095523897687


In [89]:
performance_by_year = eval_games.groupby("season").agg(
    games=("game_id", "size"), accuracy=("correct", "mean"),
    logloss=("ll", "mean")).reset_index()

performance_by_year

,season,games,accuracy,logloss
0,2023,285,0.624561,0.661789
1,2024,285,0.666667,0.608232
2,2025,284,0.654930,0.631872
3,2026,2,0.500000,0.695048


### Tuned Parameter Selection


In [91]:
params = nfl_elo["K_by_year"].copy()
params[["eval_year", "K_best", "home_adv_best", "carry_best", "mov_scale", "logloss"]]

,eval_year,K_best,home_adv_best,carry_best,mov_scale,logloss
0,2023,50.0,26.0,0.65,2.211741,0.635967
1,2024,41.0,35.0,0.65,2.211388,0.640056
2,2025,50.0,35.0,0.65,2.212730,0.635347
3,2026,50.0,35.0,0.65,2.209776,0.634906


In [93]:
print("K values:")
print(params["K_best"].value_counts().sort_index())

print("\nHome advantage values:")
print(params["home_adv_best"].value_counts().sort_index())

print("\nCarry values:")
print(params["carry_best"].value_counts().sort_index())
params[["K_best", "home_adv_best", "carry_best"]].describe()

K values:
K_best
41.0    1
50.0    3
Name: count, dtype: int64

Home advantage values:
home_adv_best
26.0    1
35.0    3
Name: count, dtype: int64

Carry values:
carry_best
0.65    4
Name: count, dtype: int64


,K_best,home_adv_best,carry_best
count,4.00,4.00,4.00
mean,47.75,32.75,0.65
std,4.50,4.50,0.00
min,41.00,26.00,0.65
25%,47.75,32.75,0.65
50%,50.00,35.00,0.65
75%,50.00,35.00,0.65
max,50.00,35.00,0.65


### Rerun Elo Tuning (Tightened Grids)

In [106]:
nfl_elo = run_expanding_k_elo(all_games=elo_df, first_train_year=2018,
                    first_eval_year=2023,
                    K_grid = compress_exp_grid(35, 60, n=7, curve=1),
                    home_adv_grid = compress_exp_grid(20, 40, n=7, curve=1),
                    scale=400, start_rating=1000, use_carry_grid=True,
                    carryover_grid = [0.5, 0.65, 0.7],
                    param_tune_window=None, use_mov=True, use_home_adv=True)

In [107]:
eval_games = nfl_elo["games"].query("home_score != away_score").copy()
eval_games["actual"] = (eval_games["home_score"] > eval_games["away_score"]).astype(int)
eval_games["pred"] = (eval_games["p_home"] >= 0.5).astype(int)
eval_games["correct"] = eval_games["pred"] == eval_games["actual"]
eval_games["ll"] = -(eval_games["actual"] * np.log(eval_games["p_home"]) + (1 - eval_games["actual"]) * np.log1p(-eval_games["p_home"]))

print("Accuracy:", accuracy_score(eval_games["actual"], eval_games["pred"]))
print("Log Loss:", log_loss(eval_games["actual"], eval_games["p_home"]))

performance_by_year = eval_games.groupby("season").agg(
    games=("game_id", "size"), accuracy=("correct", "mean"),
    logloss=("ll", "mean")).reset_index()

performance_by_year

Accuracy: 0.6530373831775701
Log Loss: 0.6338076761684792


,season,games,accuracy,logloss
0,2023,285,0.628070,0.661042
1,2024,285,0.677193,0.607362
2,2025,284,0.654930,0.632594
3,2026,2,0.500000,0.693801


In [108]:
params = nfl_elo["K_by_year"].copy()
params[["eval_year", "K_best", "home_adv_best", "carry_best", "mov_scale", "logloss"]]

,eval_year,K_best,home_adv_best,carry_best,mov_scale,logloss
0,2023,49.0,28.0,0.65,2.211741,0.635912
1,2024,44.0,31.0,0.65,2.211388,0.639933
2,2025,44.0,31.0,0.65,2.212730,0.635136
3,2026,49.0,31.0,0.65,2.209776,0.634783


## 8. Current Elo Ratings

Rebuild the latest Elo state using  selected prospectively parameters for the current season and rank all teams by their current Elo rating.

In [120]:
current_season = nfl_elo["K_by_year"]["eval_year"].max()
params = nfl_elo["K_by_year"].set_index("eval_year").loc[current_season]
ratings = {}

for s in sorted(elo_df.loc[elo_df["season"] <= current_season, "season"].unique()):
    season_games = elo_df[elo_df["season"] == s].sort_values(["gameday", "gametime", "game_id"]).reset_index(drop=True)

    res = elo_season(
        season_games, K=params["K_best"], ratings=ratings, scale=400, start_rating=1000,
        use_mov=True, mov_scale=params["mov_scale"], use_home_adv=True, home_adv=params["home_adv_best"]
    )

    ratings = res["ratings"]
    if s < current_season:
        ratings = carry_over(ratings, params["carry_best"])

current_teams = pd.unique(games.loc[games["season"] == current_season, ["home_team", "away_team"]].values.ravel())

current_ratings = pd.Series(ratings, name="elo").rename_axis("team").reset_index()
current_ratings = current_ratings.query("team in @current_teams").sort_values("elo", ascending=False).reset_index(drop=True)
current_ratings.insert(0, "rank", range(1, len(current_ratings) + 1))

current_ratings

,rank,team,elo
0,1,SEA,1190.395647
1,2,DEN,1106.535721
2,3,BUF,1104.107032
3,4,HOU,1103.879055
4,5,SF,1097.182968
5,6,NE,1091.529856
6,7,JAX,1070.937434
7,8,LA,1062.682954
8,9,MIN,1053.747206
9,10,PHI,1053.338711


## 9. Daily Rankings and Projections

Use current Elo ratings to identify games scheduled for a selected date and calculate each team's pregame Elo and win probability.

In [127]:
as_of_date = pd.Timestamp.now(tz="America/Chicago").date()
rank_map = current_ratings.set_index("team")["rank"]

future_projections = games[
    (games["season"] == current_season) &
    games["game_type"].isin(["REG", "WC", "DIV", "CON", "SB"]) &
    games["home_score"].isna() &
    games["away_score"].isna()
].copy()

future_projections["home_elo"] = future_projections["home_team"].map(ratings)
future_projections["away_elo"] = future_projections["away_team"].map(ratings)
future_projections["home_rank"] = future_projections["home_team"].map(rank_map)
future_projections["away_rank"] = future_projections["away_team"].map(rank_map)

future_projections["p_home"] = 1 / (1 + 10 ** (
    (future_projections["away_elo"] - future_projections["home_elo"] - params["home_adv_best"]) / 400
))
future_projections["p_away"] = 1 - future_projections["p_home"]
future_projections["as_of_date"] = as_of_date

future_projections = future_projections[
    ["as_of_date", "week", "gameday", "gametime", "away_team", "home_team",
     "away_rank", "home_rank", "away_elo", "home_elo", "p_away", "p_home"]
].sort_values(["gameday", "gametime"]).reset_index(drop=True)

daily_projections = future_projections[
    pd.to_datetime(future_projections["gameday"]).dt.date == as_of_date
].reset_index(drop=True)

current_week = future_projections.iloc[0]["week"]
weekly_projections = future_projections[
    future_projections["week"] == current_week
].reset_index(drop=True)

daily_rankings = current_ratings.assign(as_of_date=as_of_date)

display(future_projections)    # every remaining game using current Elo
display(daily_projections)     # today's games
display(weekly_projections)    # current/upcoming NFL week
display(daily_rankings)        # current league-wide rankings

,as_of_date,week,gameday,gametime,away_team,home_team,away_rank,home_rank,away_elo,home_elo,p_away,p_home
0,2026-09-11,1,2026-09-13,13:00,CHI,CAR,13,27,1031.790937,926.539784,0.605258,0.394742
1,2026-09-11,1,2026-09-13,13:00,TB,CIN,20,19,972.246586,975.508936,0.450852,0.549148
2,2026-09-11,1,2026-09-13,13:00,NO,DET,25,11,941.267461,1042.864220,0.317933,0.682067
3,2026-09-11,1,2026-09-13,13:00,BUF,HOU,3,4,1104.107032,1103.879055,0.455831,0.544169
4,2026-09-11,1,2026-09-13,13:00,BAL,IND,12,22,1037.779731,960.716829,0.565904,0.434096
...,...,...,...,...,...,...,...,...,...,...,...,...
265,2026-09-11,18,2027-01-10,13:00,CHI,MIN,13,9,1031.790937,1053.747206,0.424375,0.575625
266,2026-09-11,18,2027-01-10,13:00,MIA,NE,24,6,949.304377,1091.529856,0.269499,0.730501
267,2026-09-11,18,2027-01-10,13:00,TB,NO,20,25,972.246586,941.267461,0.499970,0.500030
268,2026-09-11,18,2027-01-10,13:00,PHI,NYG,10,26,1053.338711,933.364938,0.625316,0.374684


,as_of_date,week,gameday,gametime,away_team,home_team,away_rank,home_rank,away_elo,home_elo,p_away,p_home


,as_of_date,week,gameday,gametime,away_team,home_team,away_rank,home_rank,away_elo,home_elo,p_away,p_home
0,2026-09-11,1,2026-09-13,13:00,CHI,CAR,13,27,1031.790937,926.539784,0.605258,0.394742
1,2026-09-11,1,2026-09-13,13:00,TB,CIN,20,19,972.246586,975.508936,0.450852,0.549148
2,2026-09-11,1,2026-09-13,13:00,NO,DET,25,11,941.267461,1042.864220,0.317933,0.682067
3,2026-09-11,1,2026-09-13,13:00,BUF,HOU,3,4,1104.107032,1103.879055,0.455831,0.544169
4,2026-09-11,1,2026-09-13,13:00,BAL,IND,12,22,1037.779731,960.716829,0.565904,0.434096
5,2026-09-11,1,2026-09-13,13:00,CLE,JAX,28,7,921.597958,1070.937434,0.261513,0.738487
6,2026-09-11,1,2026-09-13,13:00,ATL,PIT,17,16,982.209310,1007.330636,0.419930,0.580070
7,2026-09-11,1,2026-09-13,13:00,NYJ,TEN,30,31,858.983941,858.505137,0.456189,0.543811
8,2026-09-11,1,2026-09-13,16:25,ARI,LAC,29,15,893.004384,1019.126700,0.288133,0.711867
9,2026-09-11,1,2026-09-13,16:25,MIA,LV,24,32,949.304377,848.666311,0.598896,0.401104


,rank,team,elo,as_of_date
0,1,SEA,1190.395647,2026-09-11
1,2,DEN,1106.535721,2026-09-11
2,3,BUF,1104.107032,2026-09-11
3,4,HOU,1103.879055,2026-09-11
4,5,SF,1097.182968,2026-09-11
5,6,NE,1091.529856,2026-09-11
6,7,JAX,1070.937434,2026-09-11
7,8,LA,1062.682954,2026-09-11
8,9,MIN,1053.747206,2026-09-11
9,10,PHI,1053.338711,2026-09-11


In [ ]:
### Save Data
current_ratings.to_csv("current_elo_rankings.csv", index=False)
daily_projections.to_csv("daily_projections.csv", index=False)
weekly_projections.to_csv("weekly_projections.csv", index=False)
future_projections.to_csv("future_projections.csv", index=False)
historical_elo.to_csv("nfl_elo_history.csv", index=False)